In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path

ROOT = Path(r"c:\Ground water")
DATA = ROOT / "data" / "processed"

df = pd.read_csv(DATA / "dataset_v2_3.csv", dtype={"SITE_ID": str}, low_memory=False)
with open(DATA / "feature_groups_v2_3.json") as f:
    fg = json.load(f)

TARGET = fg["target"]
tr = df[df.SPLIT == "train"]; va = df[df.SPLIT == "val"]; te = df[df.SPLIT == "test"]
print(df.shape, len(tr), len(va), len(te))

(92452, 77) 50363 1859 5178


In [1]:
import os, sys
import numpy as np
os.chdir(r"C:\Ground water")
sys.path.append(r"C:\Ground water\scripts")
import model_utils as mu

d = mu.load_data()
train, val, test = mu.split(d)
lag = "water_level_lag_1_obs_m"
target = "WLS_WTR_LEVEL"

for name, part in [("val", val), ("test", test)]:
    print("=====", name, "| rows:", len(part), "| lag NaN %:", round(part[lag].isna().mean() * 100, 1))
    print("persistence:", mu.persistence_scores(part))
    mean_pred = np.full(len(part), train[target].mean())
    print("train-mean baseline:", mu.scores(part[target], mean_pred))
    

===== val | rows: 1859 | lag NaN %: 0.0
persistence: {'n': 1859, 'R2': 0.9737, 'MAE': 2.07, 'MedAE': 1.02, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean baseline: {'n': 1859, 'R2': -0.002, 'MAE': 17.545, 'MedAE': 15.203}
===== test | rows: 5178 | lag NaN %: 0.0
persistence: {'n': 5178, 'R2': 0.9403, 'MAE': 2.648, 'MedAE': 1.155, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean baseline: {'n': 5178, 'R2': -0.0003, 'MAE': 17.311, 'MedAE': 15.143}


In [2]:
import os, sys
import numpy as np
os.chdir(r"C:\Ground water")
sys.path.append(r"C:\Ground water\scripts")
import model_utils as mu

d = mu.load_data()
train, val, test = mu.split(d)
target = "WLS_WTR_LEVEL"
train_mean = train[target].mean()
print("train mean level:", round(train_mean, 3))

for name, part in [("val", val), ("test", test)]:
    print("=====", name, "| rows:", len(part))
    print("persistence  :", mu.persistence_scores(part))
    print("train-mean   :", mu.scores(part[target], np.full(len(part), train_mean)))

train mean level: 21.243
===== val | rows: 1859
persistence  : {'n': 1859, 'R2': 0.9737, 'MAE': 2.07, 'MedAE': 1.02, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean   : {'n': 1859, 'R2': -0.002, 'MAE': 17.545, 'MedAE': 15.203}
===== test | rows: 5178
persistence  : {'n': 5178, 'R2': 0.9403, 'MAE': 2.648, 'MedAE': 1.155, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean   : {'n': 5178, 'R2': -0.0003, 'MAE': 17.311, 'MedAE': 15.143}


In [3]:
cols = [c for c in d.columns if any(k in c.lower() for k in ["date", "month", "round", "season", "period"])]
print(cols)
print(test[cols].head())

['DATE', 'MONTH', 'ROUND', 'month_sin', 'month_cos', 'ASSESSMENT_DATE', 'level_same_round_prev_year_m']
           DATE  MONTH ROUND  month_sin  month_cos ASSESSMENT_DATE  \
122  2022-01-30      1   Jan   0.500000   0.866025      2023-03-31   
123  2022-05-30      5   May   0.500000  -0.866025      2023-03-31   
124  2022-08-30      8   Aug  -0.866025  -0.500000      2023-03-31   
125  2022-11-10     11   Nov  -0.500000   0.866025      2023-03-31   
126  2023-01-10      1   Jan   0.500000   0.866025      2023-03-31   

     level_same_round_prev_year_m  
122                          4.98  
123                           NaN  
124                          4.73  
125                          5.23  
126                          6.63  


In [4]:
import json, sklearn
import numpy as np

lag = "water_level_lag_1_obs_m"
target = "WLS_WTR_LEVEL"

def by_round(part):
    out = {}
    for r in ["Jan", "May", "Aug", "Nov"]:
        p = part[part["ROUND"] == r]
        out[r] = mu.persistence_scores(p) if len(p) > 0 else {"n": 0}
    return out

res = {
    "dataset_version": "dataset_v2_3.csv",
    "rows_total": int(len(d)),
    "split_rows": {"train": int(len(train)), "val": int(len(val)), "test": int(len(test))},
    "split_years": {"train": "2005-2020", "val": "2021", "test": "2022-2023"},
    "random_state": 42,
    "sklearn_version": sklearn.__version__,
    "train_mean_level": round(float(train[target].mean()), 3),
    "persistence": {"val": mu.persistence_scores(val), "test": mu.persistence_scores(test)},
    "train_mean_baseline": {
        "val": mu.scores(val[target], np.full(len(val), train[target].mean())),
        "test": mu.scores(test[target], np.full(len(test), train[target].mean()))},
    "persistence_by_round": {"val": by_round(val), "test": by_round(test)},
}

os.makedirs(r"C:\Ground water\docs\results", exist_ok=True)
with open(r"C:\Ground water\docs\results\metrics_baseline.json", "w") as f:
    json.dump(res, f, indent=2)

for split in ["val", "test"]:
    print("=====", split, "round-wise persistence")
    for r, s in res["persistence_by_round"][split].items():
        print(r, s)

===== val round-wise persistence
Jan {'n': 617, 'R2': 0.9748, 'MAE': 1.79, 'MedAE': 0.79, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 83, 'R2': 0.9751, 'MAE': 2.922, 'MedAE': 2.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 560, 'R2': 0.9654, 'MAE': 2.452, 'MedAE': 1.3, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 598, 'R2': 0.9774, 'MAE': 1.887, 'MedAE': 0.805, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
===== test round-wise persistence
Jan {'n': 1296, 'R2': 0.9526, 'MAE': 2.182, 'MedAE': 0.8, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 1295, 'R2': 0.9376, 'MAE': 2.762, 'MedAE': 1.35, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 1284, 'R2': 0.9203, 'MAE': 3.386, 'MedAE': 1.82, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 1279, 'R2': 0.9597, 'MAE': 2.176, 'MedAE': 1.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}


In [6]:
import json, sklearn
import numpy as np

target = "WLS_WTR_LEVEL"

def by_round(part):
    out = {}
    for r in ["Jan", "May", "Aug", "Nov"]:
        p = part[part["ROUND"] == r]
        out[r] = mu.persistence_scores(p) if len(p) > 0 else {"n": 0}
    return out

res = {
    "dataset_version": "dataset_v2_3.csv",
    "rows_total": int(len(d)),
    "split_rows": {"train": int(len(train)), "val": int(len(val)), "test": int(len(test))},
    "split_years": {"train": "2005-2020", "val": "2021", "test": "2022-2023"},
    "random_state": 42,
    "sklearn_version": sklearn.__version__,
    "train_mean_level": round(float(train[target].mean()), 3),
    "persistence": {"val": mu.persistence_scores(val), "test": mu.persistence_scores(test)},
    "train_mean_baseline": {
        "val": mu.scores(val[target], np.full(len(val), train[target].mean())),
        "test": mu.scores(test[target], np.full(len(test), train[target].mean()))},
    "persistence_by_round": {"val": by_round(val), "test": by_round(test)},
}

os.makedirs(r"C:\Ground water\docs\results", exist_ok=True)
with open(r"C:\Ground water\docs\results\metrics_baseline.json", "w") as f:
    json.dump(res, f, indent=2)

for split in ["val", "test"]:
    print("=====", split, "round-wise persistence")
    for r, s in res["persistence_by_round"][split].items():
        print(r, s)

===== val round-wise persistence
Jan {'n': 617, 'R2': 0.9748, 'MAE': 1.79, 'MedAE': 0.79, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 83, 'R2': 0.9751, 'MAE': 2.922, 'MedAE': 2.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 560, 'R2': 0.9654, 'MAE': 2.452, 'MedAE': 1.3, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 598, 'R2': 0.9774, 'MAE': 1.887, 'MedAE': 0.805, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
===== test round-wise persistence
Jan {'n': 1296, 'R2': 0.9526, 'MAE': 2.182, 'MedAE': 0.8, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 1295, 'R2': 0.9376, 'MAE': 2.762, 'MedAE': 1.35, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 1284, 'R2': 0.9203, 'MAE': 3.386, 'MedAE': 1.82, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 1279, 'R2': 0.9597, 'MAE': 2.176, 'MedAE': 1.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}


In [7]:
g = mu.load_groups()
for k, v in g.items():
    print(k, "->", len(v), v[:6])

target -> 13 WLS_WT
baseline -> 6 ['DISTRICT_NAME', 'LAT', 'LON', 'YEAR', 'month_sin', 'month_cos']
rainfall -> 11 ['rainfall_mm', 'rainfall_lag_1m_mm', 'rainfall_lag_3m_mm', 'rainfall_lag_12m_mm', 'rainfall_normal_train_mm', 'rainfall_anomaly_mm']
elevation -> 1 ['elevation_m']
well_aquifer -> 2 ['WELL_TYPE', 'AQUIFER_TYPE']
lag -> 9 ['water_level_lag_1_obs_m', 'water_level_lag_2_obs_m', 'water_level_prev3_obs_mean_m', 'water_level_prev5_obs_mean_m', 'days_since_previous_observation', 'level_trend_lag1_lag2_m']
neighbor -> 3 ['neighbor_prev_level_mean_5_m', 'nearest_neighbor_prev_distance_km', 'neighbor_prev_level_count_5']
stage_2023_snapshot -> 2 ['STAGE_EXTRACTION_PCT', 'CATEGORY_CODE']
never_use_as_features -> 3 ['WELL_DEPTH', 'LEVEL_DIFF', 'OUTLIER']
ids_and_bookkeeping -> 22 ['SITE_ID', 'STATION_NAME', 'DATE', 'MONTH', 'STATE_NAME', 'SOURCE_DS']
stage_district_2023_snapshot -> 3 ['district_stage_mean_pct', 'district_overexploited_share', 'district_n_blocks']
unverified_do_not_us

In [8]:
from sklearn.ensemble import RandomForestRegressor

target = "WLS_WTR_LEVEL"
print("variant_B_groups:", g["variant_B_groups"])
print("variant_A_groups:", g["variant_A_groups"])

cols_with_year = mu.cols_for(g, ["baseline"])
cols_no_year = [c for c in cols_with_year if c != "YEAR"]

for label, cols in [("baseline, YEAR ke saath", cols_with_year), ("baseline, YEAR ke bina", cols_no_year)]:
    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    pipe = mu.make_pipeline(model, cols, train)
    pipe.fit(train[cols], train[target])
    print(label, "| features:", len(cols), "|", mu.scores(val[target], pipe.predict(val[cols])))

variant_B_groups: ['baseline', 'rainfall', 'elevation', 'well_aquifer', 'neighbor', 'stage_2023_snapshot', 'stage_district_2023_snapshot']
variant_A_groups: ['baseline', 'rainfall', 'elevation', 'well_aquifer', 'neighbor', 'stage_2023_snapshot', 'stage_district_2023_snapshot', 'lag']
baseline, YEAR ke saath | features: 6 | {'n': 1859, 'R2': 0.9759, 'MAE': 2.15, 'MedAE': 1.205}
baseline, YEAR ke bina | features: 5 | {'n': 1859, 'R2': 0.9448, 'MAE': 3.411, 'MedAE': 2.042}


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)

cols_with_year = mu.cols_for(g, ["baseline"])
cols_no_year = [c for c in cols_with_year if c != "YEAR"]

for label, cols in [("baseline, YEAR ke saath", cols_with_year), ("baseline, YEAR ke bina", cols_no_year)]:
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    pipe = mu.make_pipeline(model, cols, train)
    oof = cross_val_predict(pipe, train[cols], train[target], cv=cv, groups=groups)
    print("NAYA STATION |", label, "|", mu.scores(train[target], oof))

NAYA STATION | baseline, YEAR ke saath | {'n': 50363, 'R2': 0.5442, 'MAE': 8.602, 'MedAE': 4.67}
NAYA STATION | baseline, YEAR ke bina | {'n': 50363, 'R2': 0.5294, 'MAE': 8.88, 'MedAE': 5.0}


In [10]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)
forbidden = set(g["never_use_as_features"]) | {target}

rain_core_names = ["rainfall_mm", "rainfall_lag_1m_mm", "rainfall_lag_3m_mm", "rainfall_lag_12m_mm", "rainfall_normal_train_mm", "rainfall_anomaly_mm"]
rain_core = [c for c in g["rainfall"] if c in rain_core_names]
rain_new5 = [c for c in g["rainfall"] if c not in rain_core_names]
print("rain_core:", rain_core)
print("rain_new5:", rain_new5)

base = list(g["baseline"])
steps = [
    ("baseline", base),
    ("+rain_core", rain_core),
    ("+rain_new5", rain_new5),
    ("+elevation", list(g["elevation"])),
    ("+well_aquifer", list(g["well_aquifer"])),
    ("+neighbor", list(g["neighbor"])),
    ("+stage_district", list(g["stage_district_2023_snapshot"])),
]
extras = [
    ("+stage_block (alag row)", list(g["stage_2023_snapshot"])),
    ("+climate_extra (alag row)", list(g["climate_extra"])),
    ("+evap_soil_extra (alag row)", list(g["evap_soil_extra"])),
]

def run(label, cols):
    assert not (set(cols) & forbidden), "forbidden column mila!"
    model = RandomForestRegressor(n_estimators=60, random_state=42, n_jobs=-1)
    pipe = mu.make_pipeline(model, cols, train)
    oof = cross_val_predict(pipe, train[cols], train[target], cv=cv, groups=groups)
    s = mu.scores(train[target], oof)
    print(label, "| features:", len(cols), "|", s, flush=True)
    return {"step": label, "n_features": len(cols), "R2": s["R2"], "MAE": s["MAE"], "MedAE": s["MedAE"]}

rows, cur = [], []
for label, add in steps:
    cur = cur + add
    rows.append(run(label, cur))
full = list(cur)
for label, add in extras:
    rows.append(run(label, full + add))

os.makedirs(r"C:\Ground water\docs\results", exist_ok=True)
pd.DataFrame(rows).to_csv(r"C:\Ground water\docs\results\ablation_table.csv", index=False)
print("saved ablation_table.csv")

rain_core: ['rainfall_mm', 'rainfall_lag_1m_mm', 'rainfall_lag_3m_mm', 'rainfall_lag_12m_mm', 'rainfall_normal_train_mm', 'rainfall_anomaly_mm']
rain_new5: ['last_monsoon_total_mm', 'last_monsoon_anomaly_mm', 'rain_cum_6m_mm', 'rain_cum_12m_mm', 'rain_cum_12m_anomaly_mm']
baseline | features: 6 | {'n': 50363, 'R2': 0.5409, 'MAE': 8.605, 'MedAE': 4.646}
+rain_core | features: 12 | {'n': 50363, 'R2': 0.5404, 'MAE': 8.604, 'MedAE': 4.653}
+rain_new5 | features: 17 | {'n': 50363, 'R2': 0.5463, 'MAE': 8.529, 'MedAE': 4.594}
+elevation | features: 18 | {'n': 50363, 'R2': 0.6152, 'MAE': 8.229, 'MedAE': 4.842}
+well_aquifer | features: 20 | {'n': 50363, 'R2': 0.622, 'MAE': 8.121, 'MedAE': 4.856}
+neighbor | features: 23 | {'n': 50363, 'R2': 0.6661, 'MAE': 7.68, 'MedAE': 4.519}
+stage_district | features: 26 | {'n': 50363, 'R2': 0.67, 'MAE': 7.646, 'MedAE': 4.509}
+stage_block (alag row) | features: 28 | {'n': 50363, 'R2': 0.6624, 'MAE': 7.753, 'MedAE': 4.578}
+climate_extra (alag row) | featur

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)
forbidden = set(g["never_use_as_features"]) | {target}

colsA = mu.cols_for(g, g["variant_A_groups"])
assert not (set(colsA) & forbidden), "forbidden column mila!"
print("Variant A features:", len(colsA))

model = RandomForestRegressor(n_estimators=60, random_state=42, n_jobs=-1)
pipe = mu.make_pipeline(model, colsA, train)
oof = cross_val_predict(pipe, train[colsA], train[target], cv=cv, groups=groups)

ok = train["water_level_lag_1_obs_m"].notna().values
print("train rows lag NaN:", int((~ok).sum()))
print("Variant A (naya station, sab rows)   :", mu.scores(train[target], oof))
print("Variant A (naya station, lag wali rows):", mu.scores(train.loc[ok, target], oof[ok]))
print("persistence (train, same rows)       :", mu.persistence_scores(train))

Variant A features: 37
train rows lag NaN: 620
Variant A (naya station, sab rows)   : {'n': 50363, 'R2': 0.9599, 'MAE': 2.197, 'MedAE': 1.176}
Variant A (naya station, lag wali rows): {'n': 49743, 'R2': 0.9668, 'MAE': 2.091, 'MedAE': 1.155}
persistence (train, same rows)       : {'n': 49743, 'R2': 0.9538, 'MAE': 2.327, 'MedAE': 1.04, 'rows_dropped_lag_nan': 620, 'dropped_pct': 1.23}


In [12]:
import time
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)
forbidden = set(g["never_use_as_features"]) | {target}

featB = (list(g["baseline"]) + rain_core + rain_new5 + list(g["elevation"])
         + list(g["well_aquifer"]) + list(g["neighbor"]) + list(g["stage_district_2023_snapshot"]))
assert not (set(featB) & forbidden), "forbidden column mila!"
print("featB features:", len(featB))

model = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                     tree_method="hist", random_state=42, n_jobs=-1)
pipe = mu.make_pipeline(model, featB, train)

t0 = time.time()
oof = cross_val_predict(pipe, train[featB], train[target], cv=cv, groups=groups)
print("XGB Variant B (naya station):", mu.scores(train[target], oof))
print("time (sec):", round(time.time() - t0, 1))

featB features: 26
XGB Variant B (naya station): {'n': 50363, 'R2': 0.6805, 'MAE': 7.784, 'MedAE': 4.927}
time (sec): 7.8


In [13]:
import json, time
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)

model = XGBRegressor(tree_method="hist", random_state=42, n_jobs=-1)
pipe = mu.make_pipeline(model, featB, train)
step = pipe.steps[-1][0]
print("model step name:", step)

space = {
    f"{step}__n_estimators": [200, 400, 600],
    f"{step}__learning_rate": [0.03, 0.05, 0.1],
    f"{step}__max_depth": [4, 6, 8],
    f"{step}__min_child_weight": [1, 5, 10],
    f"{step}__subsample": [0.7, 0.85, 1.0],
    f"{step}__colsample_bytree": [0.6, 0.8, 1.0],
    f"{step}__reg_lambda": [1, 5, 10],
}

search = RandomizedSearchCV(pipe, space, n_iter=25, scoring="neg_mean_absolute_error",
                            cv=cv, random_state=42, n_jobs=1, verbose=1)
t0 = time.time()
search.fit(train[featB], train[target], groups=groups)
print("time (min):", round((time.time() - t0) / 60, 1))
print("best CV MAE:", round(-search.best_score_, 3))
print("best params:", search.best_params_)

res = pd.DataFrame(search.cv_results_)
res["MAE"] = -res["mean_test_score"]
print(res.sort_values("MAE")[["MAE"] + [c for c in res.columns if c.startswith("param_")]].head(5).to_string())

with open(r"C:\Ground water\docs\results\xgb_tuning.json", "w") as f:
    json.dump({"best_cv_mae": round(float(-search.best_score_), 3),
               "best_params": {k: (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()},
               "n_iter": 25, "cv": "GroupKFold(5) by SITE_ID", "features": featB,
               "random_state": 42}, f, indent=2)
print("saved xgb_tuning.json")

model step name: model
Fitting 5 folds for each of 25 candidates, totalling 125 fits
time (min): 4.4
best CV MAE: 7.598
best params: {'model__subsample': 0.85, 'model__reg_lambda': 5, 'model__n_estimators': 200, 'model__min_child_weight': 1, 'model__max_depth': 8, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.6}
         MAE  param_model__subsample  param_model__reg_lambda  param_model__n_estimators  param_model__min_child_weight  param_model__max_depth  param_model__learning_rate  param_model__colsample_bytree
16  7.597870                    0.85                        5                        200                              1                       8                        0.05                            0.6
19  7.599716                    1.00                        5                        200                              1                       8                        0.05                            0.6
15  7.601616                    0.85                        5   

In [14]:
import json, time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
pipe = mu.make_pipeline(model, featB, train)
step = pipe.steps[-1][0]

space = {
    f"{step}__max_depth": [12, 20, 30, None],
    f"{step}__min_samples_leaf": [1, 3, 5],
    f"{step}__max_features": [0.4, 0.6, 0.8, 1.0],
}

search_rf = RandomizedSearchCV(pipe, space, n_iter=10, scoring="neg_mean_absolute_error",
                               cv=cv, random_state=42, n_jobs=1, verbose=1)
t0 = time.time()
search_rf.fit(train[featB], train[target], groups=groups)
print("time (min):", round((time.time() - t0) / 60, 1))
print("best CV MAE (RF):", round(-search_rf.best_score_, 3))
print("best params (RF):", search_rf.best_params_)

with open(r"C:\Ground water\docs\results\rf_tuning.json", "w") as f:
    json.dump({"best_cv_mae": round(float(-search_rf.best_score_), 3),
               "best_params": {k: (v.item() if hasattr(v, "item") else v) for k, v in search_rf.best_params_.items()},
               "n_iter": 10, "cv": "GroupKFold(5) by SITE_ID", "features": featB,
               "random_state": 42}, f, indent=2)
print("saved rf_tuning.json")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
time (min): 4.7
best CV MAE (RF): 7.459
best params (RF): {'model__min_samples_leaf': 1, 'model__max_features': 0.4, 'model__max_depth': 20}
saved rf_tuning.json


In [15]:
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, cross_val_predict

groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)
target = "WLS_WTR_LEVEL"

rows = []
for label, s in [("RF tuned", search_rf), ("XGBoost tuned", search)]:
    est = clone(s.best_estimator_)
    oof = cross_val_predict(est, train[featB], train[target], cv=cv, groups=groups)
    sc = mu.scores(train[target], oof)
    print(label, sc, flush=True)
    rows.append({"model": label, "variant": "B", "validation": "naya station (GroupKFold 5)",
                 "n_features": len(featB), "R2": sc["R2"], "MAE": sc["MAE"], "MedAE": sc["MedAE"]})

pd.DataFrame(rows).to_csv(r"C:\Ground water\docs\results\comparison_table.csv", index=False)
print("saved comparison_table.csv")

RF tuned {'n': 50363, 'R2': 0.7005, 'MAE': 7.459, 'MedAE': 4.51}
XGBoost tuned {'n': 50363, 'R2': 0.6976, 'MAE': 7.598, 'MedAE': 4.768}
saved comparison_table.csv


In [16]:
import pandas as pd, numpy as np
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut, cross_val_predict

target = "WLS_WTR_LEVEL"
lagc = "water_level_lag_1_obs_m"
forbidden = set(g["never_use_as_features"]) | {target}

featA = featB + list(g["lag"])
assert not (set(featA) & forbidden), "forbidden column mila!"
print("featB:", len(featB), "| featA:", len(featA))

best_models = {"RF tuned": search_rf.best_estimator_.named_steps["model"],
               "XGBoost tuned": search.best_estimator_.named_steps["model"]}

st_groups = train["SITE_ID"].astype(str)
di_groups = train["DISTRICT_NAME"].astype(str)
ok_tr = train[lagc].notna().values
ok_va = val[lagc].notna().values

rows, dist_rows = [], []
def add(model_name, variant, validation, y, pred):
    sc = mu.scores(y, pred)
    print(model_name, "| Variant", variant, "|", validation, "|", sc, flush=True)
    rows.append({"model": model_name, "variant": variant, "validation": validation, **sc})

for model_name, base_model in best_models.items():
    for variant, cols in [("B", featB), ("A", featA)]:
        pipe = mu.make_pipeline(clone(base_model), cols, train)
        m_tr = np.ones(len(train), bool) if variant == "B" else ok_tr
        m_va = np.ones(len(val), bool) if variant == "B" else ok_va

        # 1) naya station
        oof = cross_val_predict(pipe, train[cols], train[target], cv=GroupKFold(n_splits=5), groups=st_groups)
        add(model_name, variant, "naya station (GroupKFold 5)", train.loc[m_tr, target], oof[m_tr])

        # 2) naya saal
        pipe.fit(train[cols], train[target])
        pv = pipe.predict(val[cols])
        add(model_name, variant, "naya saal (train 2005-2020 -> val 2021)", val.loc[m_va, target], pv[m_va])

        # 3) naya district
        oof_d = cross_val_predict(pipe, train[cols], train[target], cv=LeaveOneGroupOut(), groups=di_groups)
        add(model_name, variant, "naya district (leave-one-district-out)", train.loc[m_tr, target], oof_d[m_tr])

        if variant == "B":
            tmp = pd.DataFrame({"district": di_groups.values, "abs_err": np.abs(train[target].values - oof_d)})
            dd = tmp.groupby("district")["abs_err"].agg(["count", "mean"]).rename(columns={"count": "n", "mean": "MAE"})
            dd["model"] = model_name
            dist_rows.append(dd.reset_index())

add("persistence", "-", "train (lag wali rows)", train.loc[ok_tr, target], train.loc[ok_tr, lagc])
add("persistence", "-", "val 2021", val[target], val[lagc])

os.makedirs(r"C:\Ground water\docs\results", exist_ok=True)
pd.DataFrame(rows).to_csv(r"C:\Ground water\docs\results\validation_table.csv", index=False)
pd.concat(dist_rows).to_csv(r"C:\Ground water\docs\results\district_mae_variantB_lodo.csv", index=False)
print("saved validation_table.csv and district_mae_variantB_lodo.csv")

featB: 26 | featA: 35
RF tuned | Variant B | naya station (GroupKFold 5) | {'n': 50363, 'R2': 0.7005, 'MAE': 7.459, 'MedAE': 4.51}
RF tuned | Variant B | naya saal (train 2005-2020 -> val 2021) | {'n': 1859, 'R2': 0.9671, 'MAE': 2.78, 'MedAE': 1.896}
RF tuned | Variant B | naya district (leave-one-district-out) | {'n': 50363, 'R2': 0.6308, 'MAE': 8.605, 'MedAE': 5.684}
RF tuned | Variant A | naya station (GroupKFold 5) | {'n': 49743, 'R2': 0.9686, 'MAE': 2.018, 'MedAE': 1.119}
RF tuned | Variant A | naya saal (train 2005-2020 -> val 2021) | {'n': 1859, 'R2': 0.9801, 'MAE': 1.943, 'MedAE': 1.098}
RF tuned | Variant A | naya district (leave-one-district-out) | {'n': 49743, 'R2': 0.9689, 'MAE': 2.045, 'MedAE': 1.162}
XGBoost tuned | Variant B | naya station (GroupKFold 5) | {'n': 50363, 'R2': 0.6976, 'MAE': 7.598, 'MedAE': 4.768}
XGBoost tuned | Variant B | naya saal (train 2005-2020 -> val 2021) | {'n': 1859, 'R2': 0.9473, 'MAE': 3.662, 'MedAE': 2.49}
XGBoost tuned | Variant B | naya dis

In [17]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

target = "WLS_WTR_LEVEL"
lagc = "water_level_lag_1_obs_m"
st_groups = train["SITE_ID"].astype(str)
rf_base = search_rf.best_estimator_.named_steps["model"]

flag_tr = train["QC_ANY"].values
flag_va = val["QC_ANY"].values
print("train flagged:", int(flag_tr.sum()), "| val flagged:", int(flag_va.sum()))

def fit_pred(cols, tr_df, te_df):
    pipe = mu.make_pipeline(clone(rf_base), cols, tr_df)
    pipe.fit(tr_df[cols], tr_df[target])
    return pipe.predict(te_df[cols])

rows = []
for variant, cols in [("B", featB), ("A", featA)]:
    lag_ok_tr = train[lagc].notna().values if variant == "A" else np.ones(len(train), bool)
    lag_ok_va = val[lagc].notna().values if variant == "A" else np.ones(len(val), bool)
    m_tr = (~flag_tr) & lag_ok_tr
    m_va = (~flag_va) & lag_ok_va

    for label, drop_flagged in [("QC rows ke saath", False), ("QC rows ke bina", True)]:
        # naya saal (val 2021)
        tr_full = train[~train["QC_ANY"]] if drop_flagged else train
        pv = fit_pred(cols, tr_full, val)
        sc = mu.scores(val.loc[m_va, target], pv[m_va])
        print("Variant", variant, "|", label, "| naya saal |", sc, flush=True)
        rows.append({"variant": variant, "training": label, "validation": "naya saal (val 2021)", **sc})

        # naya station (GroupKFold 5, train par)
        oof = np.full(len(train), np.nan)
        for tr_i, te_i in GroupKFold(n_splits=5).split(train, groups=st_groups):
            tr_sub = train.iloc[tr_i]
            if drop_flagged:
                tr_sub = tr_sub[~tr_sub["QC_ANY"]]
            oof[te_i] = fit_pred(cols, tr_sub, train.iloc[te_i])
        sc = mu.scores(train.loc[m_tr, target], oof[m_tr])
        print("Variant", variant, "|", label, "| naya station |", sc, flush=True)
        rows.append({"variant": variant, "training": label, "validation": "naya station (GroupKFold 5)", **sc})

pd.DataFrame(rows).to_csv(r"C:\Ground water\docs\results\qc_check_table.csv", index=False)
print("saved qc_check_table.csv")

train flagged: 947 | val flagged: 22
Variant B | QC rows ke saath | naya saal | {'n': 1837, 'R2': 0.9685, 'MAE': 2.719, 'MedAE': 1.866}
Variant B | QC rows ke saath | naya station | {'n': 49416, 'R2': 0.7027, 'MAE': 7.427, 'MedAE': 4.489}
Variant B | QC rows ke bina | naya saal | {'n': 1837, 'R2': 0.968, 'MAE': 2.725, 'MedAE': 1.832}
Variant B | QC rows ke bina | naya station | {'n': 49416, 'R2': 0.7004, 'MAE': 7.461, 'MedAE': 4.489}
Variant A | QC rows ke saath | naya saal | {'n': 1837, 'R2': 0.9811, 'MAE': 1.898, 'MedAE': 1.093}
Variant A | QC rows ke saath | naya station | {'n': 48802, 'R2': 0.9721, 'MAE': 1.955, 'MedAE': 1.104}
Variant A | QC rows ke bina | naya saal | {'n': 1837, 'R2': 0.9806, 'MAE': 1.913, 'MedAE': 1.083}
Variant A | QC rows ke bina | naya station | {'n': 48802, 'R2': 0.9729, 'MAE': 1.928, 'MedAE': 1.088}
saved qc_check_table.csv
